# Media Cloud — National Collections, High-Volume Sources

This notebook:
1. Fetches all collections whose name contains **`- National`**
2. Pulls every source from those collections
3. Filters to sources where **both** `stories_per_week` and `stories_total` are in the **top 5%** globally
4. Exports the result to CSV

**Prerequisites:**  
- Free Media Cloud account → https://search.mediacloud.org/sign-up  
- Your API key (profile page after login)  
- Rate limit: 4,000 API calls/week; some endpoints capped at 2 req/min

---

In [1]:
!pip install mediacloud --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import mediacloud.api
import pandas as pd
import time
import json
import os
from pathlib import Path
from datetime import datetime
from importlib.metadata import version

print(f"mediacloud {version('mediacloud')}")
print(f"Started: {datetime.now().isoformat()}")

mediacloud 5.0.0
Started: 2026-04-24T16:35:09.353502


## 1 — API key & config

In [3]:
# ── Set your API key (pick one method) ──
# MC_API_KEY = os.environ.get("MC_API_KEY", "")
MC_API_KEY = "2c8f2e2c530f3a4e957eb517e28bb98f0f9edef8"

if not MC_API_KEY:
    raise ValueError(
        "No API key found.\n"
        "Set MC_API_KEY env var or paste it above.\n"
        "Sign up free: https://search.mediacloud.org/sign-up"
    )

mc = mediacloud.api.DirectoryApi(MC_API_KEY)
print(f"API key loaded (ends ...{MC_API_KEY[-4:]})  ✓")

API key loaded (ends ...def8)  ✓


In [22]:
# ── Configuration ──
COLLECTION_NAME_FILTER = "- National"   # substring to match in collection names
MIN_CONTENT_PER_WEEK = 900              # minimum stories/week threshold
PAGE_SIZE = 100                         # results per API page
SLEEP_BETWEEN_PAGES = 0.5               # seconds between API calls
SLEEP_BETWEEN_COLLECTIONS = 1.0         # seconds between collections
CHECKPOINT_FILE = "_national_checkpoint.json"

print(f"Will fetch collections matching: '{COLLECTION_NAME_FILTER}'")
print(f"Will keep sources with content/week > {MIN_CONTENT_PER_WEEK}")

Will fetch collections matching: '- National'
Will keep sources with content/week > 900


## 2 — Find all "- National" collections

In [5]:
def fetch_all_collections(mc, page_size=PAGE_SIZE, sleep=SLEEP_BETWEEN_PAGES):
    """Page through all collections in the directory."""
    collections = []
    offset = 0
    page = 0
    while True:
        page += 1
        try:
            resp = mc.collection_list(limit=page_size, offset=offset)
        except Exception as e:
            print(f"  ⚠ Error page {page}: {e} — retrying in 60s")
            time.sleep(60)
            continue
        batch = resp.get("results", [])
        collections.extend(batch)
        if page % 5 == 0 or resp.get("next") is None:
            print(f"  Page {page}: +{len(batch)}  (total: {len(collections)})")
        if resp.get("next") is None or len(batch) == 0:
            break
        offset += len(batch)
        time.sleep(sleep)
    return collections


print("Fetching full list of collections...")
all_collections = fetch_all_collections(mc)
print(f"\nTotal collections in Media Cloud: {len(all_collections)}")

Fetching full list of collections...
  Page 5: +100  (total: 500)
  Page 10: +100  (total: 1000)
  Page 15: +100  (total: 1500)
  Page 18: +61  (total: 1761)

Total collections in Media Cloud: 1761


In [6]:
# Filter to collections whose name contains '- National'
national_collections = [
    c for c in all_collections
    if COLLECTION_NAME_FILTER.lower() in c.get("name", "").lower()
]

df_national_colls = pd.DataFrame(national_collections)
print(f"Collections matching '{COLLECTION_NAME_FILTER}': {len(national_collections)}\n")

display_cols = [col for col in ["id", "name", "source_count", "description"]
                if col in df_national_colls.columns]
df_national_colls[display_cols]

Collections matching '- National': 243



,id,name,source_count
0,34412043,Argentina - National,1895
1,34412427,Mexico - National,647
2,38376341,Nigeria - National,440
3,34412056,Japan - National,427
4,34412356,Spain - National,376
...,...,...,...
238,34412040,Svalbard and Jan Mayen - National,1
239,34412261,Pitcairn - National,1
240,34412425,Montserrat - National,1
241,34412290,Tuvalu - National,0


In [7]:
# Save the list of national collections for reference
df_national_colls.to_csv("national_collections.csv", index=False)
print(f"Saved {len(df_national_colls)} national collections → national_collections.csv")

Saved 243 national collections → national_collections.csv


## 3 — Retrieve all sources from those collections

In [8]:
def fetch_sources_for_collection(mc, coll_id, page_size=PAGE_SIZE, sleep=SLEEP_BETWEEN_PAGES):
    """Paginate through all sources in one collection."""
    sources = []
    offset = 0
    while True:
        try:
            resp = mc.source_list(collection_id=coll_id, limit=page_size, offset=offset)
        except Exception as e:
            print(f"    ⚠ Error offset={offset}: {e} — retrying in 60s")
            time.sleep(60)
            continue
        batch = resp.get("results", [])
        sources.extend(batch)
        if resp.get("next") is None or len(batch) == 0:
            break
        offset += len(batch)
        time.sleep(sleep)
    return sources

In [11]:
import json
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

CHECKPOINT_FILE = "checkpoint.json"
max_workers = 5  # tune this (start small)

# ── Shared state ──
all_sources = {}           # source_id → source dict
source_to_collections = {} # source_id → list of collection names
done_ids = set()

lock = Lock()
t0 = time.time()

# ── Resume from checkpoint ──
if Path(CHECKPOINT_FILE).exists():
    with open(CHECKPOINT_FILE) as f:
        ckpt = json.load(f)
    all_sources = {int(k): v for k, v in ckpt.get("sources", {}).items()}
    source_to_collections = {int(k): v for k, v in ckpt.get("s2c", {}).items()}
    done_ids = set(ckpt.get("done_ids", []))

    print(f"Resumed: {len(all_sources)} sources, {len(done_ids)} collections done")

total = len(national_collections)

# ── Worker ──
def process_collection(coll):
    coll_id = coll["id"]
    coll_name = coll.get("name", "")
    try:
        sources = fetch_sources_for_collection(mc, coll_id)
    except Exception as e:
        print(f"❌ Error in {coll_name}: {e}")
        sources = []
    return coll_id, coll_name, sources

# ── Submit tasks ──
futures = []
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    for coll in national_collections:
        if coll["id"] not in done_ids:
            futures.append(executor.submit(process_collection, coll))

    completed = len(done_ids)

    # ── Process results as they complete ──
    for future in as_completed(futures):
        coll_id, coll_name, sources = future.result()

        with lock:
            for s in sources:
                sid = s.get("id")
                if sid not in all_sources:
                    all_sources[sid] = s
                    source_to_collections[sid] = []
                source_to_collections[sid].append(coll_name)

            done_ids.add(coll_id)
            completed += 1

        elapsed = time.time() - t0
        print(f"[{completed}/{total}] {coll_name} → {len(sources)} sources "
              f"| Unique total: {len(all_sources)} "
              f"| {elapsed/60:.1f} min")

        # ── Checkpoint every 20 completed collections ──
        if completed % 20 == 0:
            with open(CHECKPOINT_FILE, "w") as f:
                json.dump({
                    "sources": {str(k): v for k, v in all_sources.items()},
                    "s2c": {str(k): v for k, v in source_to_collections.items()},
                    "done_ids": list(done_ids)
                }, f)
            print("💾 Checkpoint saved")

print(f"\n✅ Done! {len(all_sources)} unique sources from {total} collections.")

[1/243] Spain - National → 376 sources | Unique total: 376 | 0.1 min
[2/243] Japan - National → 427 sources | Unique total: 788 | 0.1 min
[3/243] Nigeria - National → 440 sources | Unique total: 1169 | 0.1 min
[4/243] South Africa - National → 344 sources | Unique total: 1500 | 0.1 min
[5/243] Mexico - National → 647 sources | Unique total: 2072 | 0.1 min
[6/243] Bangladesh - National → 257 sources | Unique total: 2299 | 0.1 min
[7/243] France - National → 343 sources | Unique total: 2642 | 0.2 min
[8/243] Russian Federation - National → 243 sources | Unique total: 2871 | 0.2 min
[9/243] Colombia - National → 244 sources | Unique total: 3112 | 0.2 min
[10/243] United States - National → 248 sources | Unique total: 3357 | 0.2 min
[11/243] Norway - National → 238 sources | Unique total: 3590 | 0.2 min
[12/243] Sweden - National → 214 sources | Unique total: 3804 | 0.2 min
[13/243] China - National → 234 sources | Unique total: 3997 | 0.2 min
[14/243] Ukraine - National → 235 sources | Un

## 4 — Discover the "content per week" field name

Media Cloud documents this as "the average number of stories published per week",
but the exact JSON key varies across API versions. We inspect the first record to find it.

In [79]:
# Inspect one source record to see all available fields
sample = next(iter(all_sources.values()))
print("All fields in a source record:\n")
for k, v in sorted(sample.items()):
    print(f"  {k:30s} = {repr(v)[:80]}")

All fields in a source record:

  alternative_domains            = []
  collection_count               = 8
  created_at                     = '2022-12-23T17:43:28.547804Z'
  homepage                       = 'http://www.oem.com.mx'
  id                             = 103020
  label                          = 'OEM en Linea'
  last_rescraped                 = '2026-02-17T12:24:04.442926Z'
  last_rescraped_msg             = '0/0 added, 0/0 confirmed'
  last_story                     = '04/2026'
  media_type                     = 'print_native'
  modified_at                    = '2026-04-18T09:42:20.773689Z'
  monitored                      = True
  name                           = 'oem.com.mx'
  notes                          = 'Child source: https://oem.com.mx/elsoldemexico/'
  platform                       = 'online_news'
  primary_language               = 'es'
  pub_country                    = 'MEX'
  pub_state                      = 'MX-CMX'
  stories_date_empty             = 24858
  

In [80]:
# ── Auto-detect the stories-per-week field ──
# Known candidate names across API versions:
CANDIDATE_FIELDS = [
    "stories_per_week",
    "content_per_week",
    "story_count_per_week",
    "avg_stories_per_week",
    "num_stories_90",
    "stories_per_day",
]

cpw_field = None
for candidate in CANDIDATE_FIELDS:
    if candidate in sample:
        cpw_field = candidate
        break

# Fallback: look for any field with 'stor' or 'content' AND 'week' or 'per'
if cpw_field is None:
    for k in sample.keys():
        kl = k.lower()
        if ("stor" in kl or "content" in kl) and ("week" in kl or "per" in kl):
            cpw_field = k
            break

if cpw_field:
    print(f"✅ Using field: '{cpw_field}'")
    print(f"   Sample value: {sample[cpw_field]}")
else:
    print("⚠ Could not auto-detect the content-per-week field.")
    print("  Check the field list above and set cpw_field manually:")
    print('  cpw_field = "your_field_name_here"')

✅ Using field: 'stories_per_week'
   Sample value: 7230


## 5 — Build DataFrame, add collections column, filter

In [81]:
df_all = pd.DataFrame(all_sources.values())

# Add a column with the list of national collections each source belongs to
df_all["national_collections"] = df_all["id"].map(
    lambda sid: " | ".join(sorted(set(source_to_collections.get(sid, []))))
)

print(f"All sources (before filter): {len(df_all)}")
print(f"Columns: {list(df_all.columns)}")
df_all.head(3)

All sources (before filter): 14433
Columns: ['id', 'name', 'url_search_string', 'label', 'homepage', 'notes', 'platform', 'stories_per_week', 'last_story', 'created_at', 'modified_at', 'pub_country', 'pub_state', 'primary_language', 'media_type', 'last_rescraped', 'last_rescraped_msg', 'collection_count', 'alternative_domains', 'stories_total', 'stories_date_past', 'stories_date_future', 'stories_date_empty', 'monitored', 'national_collections']


,id,name,url_search_string,label,homepage,notes,platform,stories_per_week,last_story,created_at,...,last_rescraped,last_rescraped_msg,collection_count,alternative_domains,stories_total,stories_date_past,stories_date_future,stories_date_empty,monitored,national_collections
0,103020,oem.com.mx,None,OEM en Linea,http://www.oem.com.mx,Child source: https://oem.com.mx/elsoldemexico/,online_news,7230.0,04/2026,2022-12-23T17:43:28.547804Z,...,2026-02-17T12:24:04.442926Z,"0/0 added, 0/0 confirmed",8,[],71866.0,0.0,0.0,24858.0,True,Mexico - National
1,239478,elimparcial.com,None,"ELIMPARCIAL.COM - Noticias, Clasificados e Inf...",http://www.elimparcial.com,None,online_news,1622.0,04/2026,2022-12-23T17:43:28.547804Z,...,2026-02-25T16:38:18.313935Z,"0/1 added, 1/3 confirmed",10,[],594496.0,0.0,0.0,38.0,True,Mexico - National
2,23668,eluniversal.com.mx,None,eluniversal.com.mx,http://www.eluniversal.com.mx,None,online_news,1589.0,04/2026,2022-12-23T17:43:28.547804Z,...,2026-04-05T07:04:26.390283Z,"0/1 added, 1/2 confirmed",11,[],475018.0,88.0,0.0,168.0,True,Mexico - National


In [97]:
# ── Filter: both stories_per_week and stories_total in top 20% (globally) ──

RANK_FIELD = "stories_per_week"
SECONDARY_FIELD = "stories_total"
TOP_PCT = 0.95

df_all[RANK_FIELD] = pd.to_numeric(df_all[RANK_FIELD], errors="coerce")
df_all[SECONDARY_FIELD] = pd.to_numeric(df_all[SECONDARY_FIELD], errors="coerce")

cpw_thresh = df_all[RANK_FIELD].quantile(TOP_PCT)
total_thresh = df_all[SECONDARY_FIELD].quantile(TOP_PCT)

df_filtered = df_all[
    (df_all[RANK_FIELD] >= cpw_thresh) &
    (df_all[SECONDARY_FIELD] >= total_thresh)
].sort_values(RANK_FIELD, ascending=False).reset_index(drop=True)

print(f"80th-percentile thresholds:")
print(f"  {RANK_FIELD}:  >= {cpw_thresh:,.0f}")
print(f"  {SECONDARY_FIELD}: >= {total_thresh:,.0f}")
print(f"\nSources passing both filters: {len(df_filtered):,} / {len(df_all):,}")

80th-percentile thresholds:
  stories_per_week:  >= 744
  stories_total: >= 337,232

Sources passing both filters: 370 / 14,433


In [98]:
# Quick preview of the filtered data
preview_cols = [c for c in ["id", "name", "domain", "homepage", "url",
                            cpw_field, "pub_country", "primary_language",
                            "media_type", "national_collections"]
                if c and c in df_filtered.columns]

df_filtered[preview_cols].head(15)

,id,name,homepage,stories_per_week,pub_country,primary_language,media_type,national_collections
0,219540,bhaskar.com,https://www.bhaskar.com/,45792.0,IND,hi,print_native,India - National
1,180285,news.de,http://news.de/,26208.0,DEU,de,digital_native,Germany - National (Media Impact Monitor Test)...
2,262319,amarujala.com,https://www.amarujala.com/,23140.0,IND,hi,print_native,India - National
3,657923,fnnews.com,http://www.fnnews.com/,18852.0,KOR,ko,print_native,"Korea, Republic of - National"
4,126565,infobae.com,http://infobae.com/,15974.0,ARG,es,digital_native,Argentina - National | Colombia - National
5,115065,indiatimes.com,http://indiatimes.com/,14455.0,IND,en,digital_native,India - National
6,39784,indiatimes.com,http://timesofindia.indiatimes.com/,10213.0,IND,en,print_native,India - National
7,376850,prtimes.jp,http://prtimes.jp/,9566.0,None,ja,None,Japan - National
8,932549,bbc.com,http://bbc.com,9037.0,GBR,en,None,Afghanistan - National | Bangladesh - National...
9,18212,dostor.org,http://dostor.org,8893.0,EGY,ar,print_native,Egypt - National | Kuwait - National


## 6 — Export to CSV

In [99]:
ts = datetime.now().strftime("%Y%m%d")

filtered_file = f"national_sources_top5pct_both_{ts}.csv"
df_filtered.to_csv(filtered_file, index=False)
print(f"✅ {len(df_filtered)} sources (top-5% on both metrics) → {filtered_file}")

all_file = f"national_sources_all_{ts}.csv"
df_all.to_csv(all_file, index=False)
print(f"✅ {len(df_all)} total sources (unfiltered) → {all_file}")

✅ 370 sources (top-5% on both metrics) → national_sources_top5pct_both_20260424.csv
✅ 14433 total sources (unfiltered) → national_sources_all_20260424.csv


## 7 — Summary stats

In [100]:
print("=" * 60)
print(f"National collections found:      {len(national_collections)}")
print(f"Total unique sources (all):      {len(df_all):,}")
print(f"Sources passing top-20% filter: {len(df_filtered):,}")
for field in [RANK_FIELD, SECONDARY_FIELD]:
    if field in df_filtered.columns:
        print(f"  {field} — max: {df_filtered[field].max():,.0f}, median: {df_filtered[field].median():,.0f}")
print("=" * 60)

National collections found:      243
Total unique sources (all):      14,433
Sources passing top-20% filter: 370
  stories_per_week — max: 45,792, median: 1,530
  stories_total — max: 8,046,580, median: 644,411


In [101]:
# Country distribution in the filtered set
country_col = next(
    (c for c in ["pub_country", "country"] if c in df_filtered.columns), None
)
if country_col:
    print(f"Top 25 countries in filtered sources ('{country_col}'):\n")
    print(df_filtered[country_col].value_counts().head(25).to_string())

Top 25 countries in filtered sources ('pub_country'):

pub_country
ESP    34
USA    21
IND    16
TUR    13
GBR    11
VNM    10
UKR     9
SRB     8
MEX     7
ITA     7
BRA     7
HRV     6
RUS     6
ARG     6
AUT     5
KOR     5
GRC     5
IDN     5
XKX     5
TWN     4
DEU     4
AZE     4
BEL     3
IRN     3
HUN     3


In [102]:
# Top 30 sources in the filtered set
top_cols = [c for c in ["name", "homepage", RANK_FIELD, SECONDARY_FIELD,
                         "pub_country", "primary_language", "national_collections"]
            if c in df_filtered.columns]
print(f"Top 30 sources by {RANK_FIELD}:\n")
display(df_filtered[top_cols].head(30))

Top 30 sources by stories_per_week:



,name,homepage,stories_per_week,stories_total,pub_country,primary_language,national_collections
0,bhaskar.com,https://www.bhaskar.com/,45792.0,7313992.0,IND,hi,India - National
1,news.de,http://news.de/,26208.0,1509810.0,DEU,de,Germany - National (Media Impact Monitor Test)...
2,amarujala.com,https://www.amarujala.com/,23140.0,6048073.0,IND,hi,India - National
3,fnnews.com,http://www.fnnews.com/,18852.0,1170323.0,KOR,ko,"Korea, Republic of - National"
4,infobae.com,http://infobae.com/,15974.0,2927077.0,ARG,es,Argentina - National | Colombia - National
5,indiatimes.com,http://indiatimes.com/,14455.0,7906665.0,IND,en,India - National
6,indiatimes.com,http://timesofindia.indiatimes.com/,10213.0,4267907.0,IND,en,India - National
7,prtimes.jp,http://prtimes.jp/,9566.0,2212584.0,None,ja,Japan - National
8,bbc.com,http://bbc.com,9037.0,1118330.0,GBR,en,Afghanistan - National | Bangladesh - National...
9,dostor.org,http://dostor.org,8893.0,2332707.0,EGY,ar,Egypt - National | Kuwait - National


## 8 — Cleanup

In [103]:
if Path(CHECKPOINT_FILE).exists():
    Path(CHECKPOINT_FILE).unlink()
    print("Checkpoint removed.")

print("\n🎉 Output files:")
for f in sorted(Path(".").glob("national_*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:,.0f} KB)")


🎉 Output files:
  national_collections.csv  (33 KB)
  national_sources_all_20260424.csv  (3,863 KB)
  national_sources_cpw_gt_900_20260424.csv  (132 KB)
  national_sources_top10pct_both_20260424.csv  (219 KB)
  national_sources_top20_top3percountry_20260424.csv  (105 KB)
  national_sources_top20pct_both_20260424.csv  (219 KB)
  national_sources_top5_per_country_20260424.csv  (313 KB)
  national_sources_top5pct_both_20260424.csv  (101 KB)
